In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.nn.utils.rnn import (
    pad_sequence
)

In [ ]:
TEST_FEATURES_DIR = "/content/drive/MyDrive/preprocessed/new_features/test"

CSV_PATH = "/content/drive/MyDrive/preprocessed/oasis_tlstm_ready_cleaned.csv"

MODEL_PATH = "/content/drive/MyDrive/preprocessed/tlstm_model.pth"

PCA_COMPONENTS_PATH = "/content/drive/MyDrive/preprocessed/pca_components.npy"

PCA_MEAN_PATH = "/content/drive/MyDrive/preprocessed/pca_mean.npy"

SCALER_MEAN_PATH = "/content/drive/MyDrive/preprocessed/scaler_mean.npy"

SCALER_SCALE_PATH = "/content/drive/MyDrive/preprocessed/scaler_scale.npy"

PCA_COMPONENTS = 128

BATCH_SIZE = 8

MAX_SEQ_LEN = 3

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
CLINICAL_COLS = [
    'MMSE',
    'eTIV',
    'nWBV',
    'ASF'
]

STATIC_COLS = [
    'EDUC',
    'SES',
    'M/F_M'
]

FEATURE_COLS = (
    CLINICAL_COLS +
    STATIC_COLS
)

TARGET_COL = 'CDR'

PATIENT_COL = 'patient_id'

VISIT_COL = 'Visit'

MRI_COL = 'mri_id'

In [ ]:
df = pd.read_csv(CSV_PATH)

df.head()

,patient_id,mri_id,Group,Visit,mr_delay_raw,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF,M/F_M
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1986.550000,0.696106,0.883440,1
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004.479526,0.681062,0.875539,1
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,2.0,23.0,0.5,1678.290000,0.736336,1.045710,1
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,2.0,28.0,0.5,1737.620000,0.713402,1.010000,1
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,2.0,22.0,0.5,1697.911134,0.701236,1.033623,1


In [ ]:
scaler = StandardScaler()

scaler.mean_ = np.load(
    SCALER_MEAN_PATH
)

scaler.scale_ = np.load(
    SCALER_SCALE_PATH
)

df[FEATURE_COLS] = scaler.transform(
    df[FEATURE_COLS]
)

print("Scaler loaded")

Scaler loaded


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [ ]:
test_feature_map = {}

for root, dirs, files in os.walk(
        TEST_FEATURES_DIR):

    for file in files:

        if file.endswith(".npy"):

            image_id = file.replace(
                ".npy",
                ""
            )

            feat = np.load(
                os.path.join(root, file)
            )

            test_feature_map[
                image_id
            ] = feat

print(
    "Loaded test features:",
    len(test_feature_map)
)

Loaded test features: 76


In [ ]:
pca = PCA(
    n_components=PCA_COMPONENTS
)

pca.components_ = np.load(
    PCA_COMPONENTS_PATH
)

pca.mean_ = np.load(
    PCA_MEAN_PATH
)

print("PCA loaded")

import joblib
pca = joblib.load(
    "/content/drive/MyDrive/pca.pkl"
)

print("PCA loaded")

PCA loaded
PCA loaded


In [ ]:
test_ids = list(
    test_feature_map.keys()
)

test_features = np.stack(
    list(test_feature_map.values())
)

test_features_pca = pca.transform(
    test_features
)

print(
    "Reduced shape:",
    test_features_pca.shape
)

Reduced shape: (76, 128)


In [ ]:
test_feature_map_pca = {}

for i, img_id in enumerate(test_ids):

    test_feature_map_pca[
        img_id
    ] = test_features_pca[i]

In [ ]:
test_sequences = []

df = df.sort_values(
    [PATIENT_COL, VISIT_COL]
)

grouped = df.groupby(PATIENT_COL)

for patient_id, patient_df in grouped:

    clinical_sequence = []

    mri_sequence = []

    patient_df = patient_df.sort_values(
        VISIT_COL
    )

    for _, row in patient_df.iterrows():

        image_id = row[MRI_COL]

        if image_id not in test_feature_map_pca:
            continue

        clinical_feat = row[
            FEATURE_COLS
        ].values.astype(np.float32)

        mri_feat = test_feature_map_pca[
            image_id
        ].astype(np.float32)

        clinical_sequence.append(
            clinical_feat
        )

        mri_sequence.append(
            mri_feat
        )

    if len(clinical_sequence) < 2:
        continue

    clinical_sequence = (
        clinical_sequence[-MAX_SEQ_LEN:]
    )

    mri_sequence = (
        mri_sequence[-MAX_SEQ_LEN:]
    )

    target = np.float32(
        patient_df.iloc[-1][TARGET_COL]
    )

    test_sequences.append(
        (
            np.array(clinical_sequence),
            np.array(mri_sequence),
            target
        )
    )

print(
    "Test sequences:",
    len(test_sequences)
)

Test sequences: 30


In [ ]:
class ProgressionDataset(Dataset):

    def __init__(self, sequences):

        self.sequences = sequences

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        clinical_x, mri_x, y = (
            self.sequences[idx]
        )

        return (
            torch.tensor(
                clinical_x,
                dtype=torch.float32
            ),

            torch.tensor(
                mri_x,
                dtype=torch.float32
            ),

            torch.tensor(
                y,
                dtype=torch.float32
            )
        )

In [ ]:
def collate_fn(batch):

    clinical_x = [
        item[0] for item in batch
    ]

    mri_x = [
        item[1] for item in batch
    ]

    y = [
        item[2] for item in batch
    ]

    clinical_x = pad_sequence(
        clinical_x,
        batch_first=True,
        padding_value=0
    )

    mri_x = pad_sequence(
        mri_x,
        batch_first=True,
        padding_value=0
    )

    y = torch.tensor(
        y,
        dtype=torch.float32
    )

    return clinical_x, mri_x, y

In [ ]:
test_dataset = ProgressionDataset(
    test_sequences
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn
)

In [ ]:
class TLSTM(nn.Module):

    def __init__(self,
                 clinical_dim,
                 mri_dim,
                 hidden_dim=64,
                 dropout=0.3):

        super(TLSTM, self).__init__()

        input_dim = (
            clinical_dim + mri_dim
        )

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.regressor = nn.Sequential(

            nn.Linear(hidden_dim, 32),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(32, 1)
        )

    def forward(self,
                clinical_x,
                mri_x):

        x = torch.cat(
            [clinical_x, mri_x],
            dim=-1
        )

        lstm_out, _ = self.lstm(x)

        out = self.regressor(
            lstm_out[:, -1, :]
        )

        return out

In [ ]:
model = TLSTM(
    clinical_dim=len(FEATURE_COLS),
    mri_dim=PCA_COMPONENTS
).to(device)

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

print("Model loaded")

Model loaded


In [ ]:
all_preds = []

all_targets = []

with torch.no_grad():

    for clinical_x, mri_x, y in test_loader:

        clinical_x = clinical_x.to(device)

        mri_x = mri_x.to(device)

        preds = model(
            clinical_x,
            mri_x
        ).squeeze()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_targets.extend(
            y.numpy()
        )

all_preds = np.array(all_preds)

all_targets = np.array(all_targets)

In [ ]:
mae = mean_absolute_error(
    all_targets,
    all_preds
)

rmse = np.sqrt(
    mean_squared_error(
        all_targets,
        all_preds
    )
)

r2 = r2_score(
    all_targets,
    all_preds
)

print("\nTEST RESULTS\n")

print(f"MAE  : {mae:.4f}")

print(f"RMSE : {rmse:.4f}")

print(f"R²   : {r2:.4f}")


TEST RESULTS

MAE  : 0.2265
RMSE : 0.2946
R²   : 0.0917
